criando uma tabela de orders com: quantidade de itens, vendedores, mapeamento de status e valor total da venda.

In [0]:
%python

df_orders_silver = spark.sql(
    """
with total_venda as (
    select 
       order_id,
       round((list_price*quantity)*(1- discount),2) as total_venda--(list_price*quantity)-(list_price*discount)
       from 1bronze_bike.order_items
    )
SELECT 
o.order_id,
o.customer_id,
o.order_date,
o.required_date,
o.shipped_date,
sto.store_name,
sto.state as store_state,
sto.city as store_city,
sta.first_name as staff_first_name,
sta.email as staff_email,
sta.active as staff_active,
total_venda,
CASE
    WHEN o.order_status = 1 then 'Pending'
    WHEN o.order_status = 2 then 'Processing'
    WHEN o.order_status =3 then 'Shipped'
    WHEN o.order_status = 4 then 'Delivered'
    else 'Unknown'
    END AS Status
from 1bronze_bike.orders  as o

left join 1bronze_bike.stores as sto
on o.store_id = sto.store_id

left join 1bronze_bike.staffs as sta
on o.staff_id = sta.staff_id

left join total_venda 
on o.order_id = total_venda.order_id

                    """
)

#display(df_orders_silver)

In [0]:
df_orders_silver.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema','true')\
    .saveAsTable('2silver_bike.orders')